# Strain ↔ microstructure correlation — 111 strain-mosa (Al, 17 keV)

This notebook is a **downstream extension** of `strain_analysis.ipynb`.
It reuses the *already-corrected* axial strain field produced by that
notebook's obpitch → 2θ correction workflow and correlates it with the
microstructure extracted from the mosaicity map:

1. segmented mosaicity-cell boundaries vs strain;
2. KAM (from `darling.transforms.kam`) vs strain;
3. per-cell strain statistics vs mosaicity / KAM statistics;
4. (optional) a second 2D mosaicity dataset from the same ROI at a slightly
   different load step, to test whether high/low strain regions are the
   regions that change most.

**The 2θ / strain correction from `strain_analysis.ipynb` is preserved
verbatim** (Section 2). This notebook never rederives that correction; it
only consumes its final corrected strain map as the reference field.

## 1. Imports and configuration

In [ ]:
import sys
import os
# Local darling checkout (HPC): needed for darling.transforms.rgb/kam if
# darling is not pip-installed in the active environment. Harmless if the
# path does not exist or darling is already importable.
_DARLING_CHECKOUT = "/zhome/b3/e/209034/darling"
if os.path.isdir(_DARLING_CHECKOUT) and _DARLING_CHECKOUT not in sys.path:
    sys.path.append(_DARLING_CHECKOUT)


In [ ]:
from pathlib import Path
import json
import inspect

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage, stats
from scipy.ndimage import distance_transform_edt

import darling
import darling.transforms as dtransforms

import disell

In [ ]:
# Optional dependencies degrade gracefully.
try:
    from skimage.segmentation import find_boundaries
except ImportError:
    find_boundaries = None

try:
    from skimage.registration import phase_cross_correlation
except ImportError:
    phase_cross_correlation = None

print('find_boundaries available     :', find_boundaries is not None)
print('phase_cross_correlation avail :', phase_cross_correlation is not None)

In [ ]:
# Inspect the relevant APIs before calling them (do not guess signatures).
import disell.cell_identification as ci
print('disell.cell_identification:')
print('  ', [a for a in dir(ci) if not a.startswith('_')])
print('darling.transforms:')
print('  ', [a for a in dir(dtransforms) if not a.startswith('_')])
print('kam signature:', inspect.signature(dtransforms.kam))
print('rgb signature:', inspect.signature(dtransforms.rgb))
print('flood_fill_dfxm signature:', inspect.signature(ci.flood_fill_dfxm))

In [ ]:
# ----------------------------------------------------------------------- #
# Paths
# ----------------------------------------------------------------------- #
# DATA_DIR_1 must point at the SAME strain-mosa dataset used by
# strain_analysis.ipynb (a folder with mean.npy (Y,X,3) = phi,chi,obpitch
# and motors.npy (3,m,n,o)).
DATA_DIR_1 = Path(
    "/dtu/3d-imaging-center/projects/2022_QIM_PMP/analysis/Adam/111_june/"
    "111_cells_2_6-7pct_strainmosa_2x_redo2/strainmosa_3d"
)

# Optional second 2D mosaicity dataset (same ROI, different load step).
# Leave as a non-existent placeholder to skip the second-dataset section.
DATA_DIR_2 = Path("PUT_SECOND_2D_MOSA_DATASET_HERE")

OUT_DIR = DATA_DIR_1 / "strain_microstructure_analysis"
OUT_DIR.mkdir(exist_ok=True, parents=True)

# Channel mapping for mean.npy (phi, chi, obpitch). VERIFY against the
# printed motor ranges in Section 2 (obpitch must be the ~17.9-18.0 deg ch).
CH_PHI = 0
CH_CHI = 1
CH_OBP = 2

HAS_SECOND_DATASET = DATA_DIR_2.exists()
print('OUT_DIR             :', OUT_DIR)
print('HAS_SECOND_DATASET  :', HAS_SECOND_DATASET)

In [ ]:
# ----------------------------------------------------------------------- #
# Segmentation parameters (TUNABLE analysis parameters, not final science).
# These control the disell flood-fill cell segmentation of the mosaicity
# field. Thresholds are in the native units of the segmentation input
# (see SEG_NORMALIZE below).
# ----------------------------------------------------------------------- #
SEG_LOCAL_THRESHOLD = 0.01
SEG_GLOBAL_THRESHOLD = 0.5
SEG_MIN_SIZE = 5
SEG_RECYCLE_SMALL = True
SEG_MAX_ITERATIONS = 1_500_000
SEG_FOOTPRINT_TOLERANCE = 1.0
SEG_RANDOM_SEED = 0

# If True, per-channel min-max (1-99 pct) normalisation is applied to the
# mosaicity before segmentation so the thresholds live on a ~[0,1] scale
# (matches the disell batch scripts). If False, segmentation runs on the
# raw mosaicity in its native angular units. Default False follows the
# 'seg_input = mosa_m' convention in the task spec.
SEG_NORMALIZE = False

# KAM kernel size (rows, cols) in pixels.
KAM_SIZE = (3, 3)

## 2. Load / reproduce the existing strain-analysis workflow

The cells below are **copied verbatim** from `strain_analysis.ipynb` (only
`DATA_DIR` → `DATA_DIR_1`). They reproduce the working outputs:

* mosaicity map;
* valid mask (`valid`);
* corrected axial strain map (`strain_m`);
* Bragg-angle sanity check;
* the current strain plot.

Do **not** modify the 2θ / obpitch correction logic here. The empirical
geometric vertical de-ramp, the `RAMP_AXIS` / `OBPITCH_TO_2THETA` choices,
the reference-angle choice and the Bragg conversion are all preserved as in
the original notebook.

In [ ]:
# --- [from strain_analysis.ipynb] crystal / beam constants + correction
#     control variables. Preserved exactly.
CH_LABELS = {CH_PHI: "phi", CH_CHI: "chi", CH_OBP: "obpitch (2theta)"}

# Crystal / beam: Al (fcc) 111 at 17 keV
A_AL = 4.0495        # Angstrom, room-T lattice parameter
E_KEV = 17.0
HC = 12.39842        # keV * Angstrom

# obpitch step == 2theta step (degrees). Verified: 2theta_B ~ 17.946 deg
# matches the obpitch scan start (17.9441 deg), i.e. obpitch is 2theta in deg.
OBPITCH_TO_2THETA = 1.0

# Axis of the geometric strain ramp. Geometry predicts purely vertical (rows).
RAMP_AXIS = 0        # 0 = rows (vertical); switch to 1 only if ramp is horizontal

In [ ]:
# --- [from strain_analysis.ipynb] load inputs
mean = np.load(DATA_DIR_1 / "mean.npy")        # (Y, X, 3)
motors = np.load(DATA_DIR_1 / "motors.npy")    # (3, m, n, o)
try:
    info = json.loads((DATA_DIR_1 / "processing_info.json").read_text())
except FileNotFoundError:
    info = {}

print("mean   :", mean.shape)
print("motors :", motors.shape)
for c, lbl in CH_LABELS.items():
    print(f"  ch {c} = {lbl:18s} motor grid range [{motors[c].min():.4f}, {motors[c].max():.4f}]")
print("obpitch steps:", np.round(motors[CH_OBP, 0, 0, :], 5))

In [ ]:
# --- [from strain_analysis.ipynb] Bragg-angle sanity check
d111 = A_AL / np.sqrt(3.0)
lam = HC / E_KEV
theta_B = np.degrees(np.arcsin(lam / (2.0 * d111)))
cot_B = 1.0 / np.tan(np.radians(theta_B))

print(f"d111     = {d111:.4f} A")
print(f"lambda   = {lam:.4f} A")
print(f"theta_B  = {theta_B:.4f} deg")
print(f"2*theta_B= {2*theta_B:.4f} deg")
print(f"cot(theta_B) = {cot_B:.4f}")
print(f"obpitch range = [{motors[CH_OBP].min():.4f}, {motors[CH_OBP].max():.4f}] deg")

In [ ]:
# --- [from strain_analysis.ipynb] valid-pixel mask
obp = mean[..., CH_OBP]
obp_lo, obp_hi = motors[CH_OBP].min(), motors[CH_OBP].max()
valid = (obp >= obp_lo - 1e-6) & (obp <= obp_hi + 1e-6)
print(f"valid pixels: {valid.mean()*100:.1f} %")

In [ ]:
# --- [from strain_analysis.ipynb] raw centroid maps: phi, chi, obpitch
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (c, lbl) in zip(axes, CH_LABELS.items()):
    m = np.where(valid, mean[..., c], np.nan)
    im = ax.imshow(m, cmap="viridis")
    ax.set_title(lbl)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

In [ ]:
# --- [from strain_analysis.ipynb] empirical geometric vertical de-ramp
rr, cc = np.where(valid)
Amat = np.vstack([rr, cc, np.ones_like(rr)]).T
(srow, scol, b0), *_ = np.linalg.lstsq(Amat, obp[valid], rcond=None)

print(f"row (vertical) slope = {srow:.3e} deg/px  -> {srow*mean.shape[0]:.4f} deg across FOV")
print(f"col (horizontal) slope = {scol:.3e} deg/px  -> {scol*mean.shape[1]:.4f} deg across FOV")
print("Geometry (eq 27-28): 2theta shift is purely vertical -> expect |row| >> |col|.")

# --- empirical geometric de-ramp (default) ---
if RAMP_AXIS == 0:
    ramp = srow * np.arange(mean.shape[0])[:, None]
else:
    ramp = scol * np.arange(mean.shape[1])[None, :]

# --- theoretical alternative (uncomment + fill in to preserve real vertical gradients):
# GAMMA = 2.93           # m^-1, geometric factor (Poulsen Fig 6 value is setup-specific!)
# M_MAG = 15.1           # total magnification
# PIX_M = 0.65e-6 * 2    # effective pixel size at sample (detector pixel * bin / M)
# z_s = -(np.arange(mean.shape[0]) - mean.shape[0]/2)[:, None] * PIX_M
# ramp = -np.degrees(GAMMA * z_s)   # 2theta_shift = -gamma * z_s

obp_corr = obp - ramp
obp_corr_m = np.where(valid, obp_corr, np.nan)

In [ ]:
# --- [from strain_analysis.ipynb] ramp-removal row profile
rows = np.arange(mean.shape[0])
prof_raw = np.nanmedian(np.where(valid, obp, np.nan), axis=1)
prof_cor = np.nanmedian(obp_corr_m, axis=1)

plt.figure(figsize=(6, 5))
plt.plot(prof_raw, rows, label="raw")
plt.plot(prof_cor, rows, label="de-ramped")
plt.gca().invert_yaxis()
plt.xlabel("obpitch / 2theta (deg)"); plt.ylabel("row")
plt.title("Vertical geometric ramp removal"); plt.legend()
plt.show()

In [ ]:
# --- [from strain_analysis.ipynb] corrected 2theta -> axial strain
two_theta = obp_corr * OBPITCH_TO_2THETA                          # deg
two_theta_ref = np.nanmedian(np.where(valid, two_theta, np.nan))  # relative zero (FOV median)
dtheta = np.radians(two_theta - two_theta_ref) / 2.0              # rad
strain = -cot_B * dtheta
strain_m = np.where(valid, strain, np.nan)

print(f"reference 2theta = {two_theta_ref:.4f} deg (strain zero)")
print(f"strain rms = {np.nanstd(strain_m):.2e}")
print(f"strain 2-98 pct = [{np.nanpercentile(strain_m,2):.2e}, {np.nanpercentile(strain_m,98):.2e}]")

In [ ]:
# --- [from strain_analysis.ipynb] strain map (diverging, symmetric)
v = np.nanpercentile(np.abs(strain_m), 98)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(strain_m, cmap="RdBu_r", vmin=-v, vmax=v)
plt.colorbar(im, ax=ax, label=r"axial strain $\varepsilon = \Delta d / d$")
ax.set_title("Axial strain (geometric vertical ramp removed)")
fig.savefig(OUT_DIR / "corrected_strain.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Preserve the existing corrected strain map

We now freeze the outputs of the preserved workflow into clean, explicit
downstream aliases. The corrected strain field (`strain_m`) and the valid
mask (`valid`) are taken directly from Section 2 — **not recomputed**.

In [ ]:
# Downstream aliases. strain_m and valid already carry the existing
# corrected-strain and valid-mask definitions from Section 2.
strain_m = np.where(valid, strain_m, np.nan)
assert strain_m.shape == valid.shape
print('strain_m shape :', strain_m.shape)
print('valid pixels   :', int(valid.sum()))

## 4. Prepare the mosaicity field

Build masked phi / chi maps and the two-channel mosaicity field used for
RGB visualisation, segmentation and KAM. Invalid pixels are set to NaN.

In [ ]:
phi_m = np.where(valid, mean[..., CH_PHI].astype(float), np.nan)
chi_m = np.where(valid, mean[..., CH_CHI].astype(float), np.nan)

mosa_m = np.stack([phi_m, chi_m], axis=-1)
mosa_m[~valid] = np.nan

assert strain_m.shape == valid.shape
assert phi_m.shape == valid.shape
assert chi_m.shape == valid.shape
assert mosa_m.shape[:2] == valid.shape
assert mosa_m.shape[-1] == 2
print('mosa_m shape :', mosa_m.shape)

## 5. Mosaicity RGB visualisation (`darling.transforms.rgb`)

Same call convention as `strain_analysis.ipynb`: `norm="full"` with the
phi/chi motor grid as `coordinates`, so colours are comparable across
datasets sharing the same mosa grid.

In [ ]:
rgb_map, colorkey, colorgrid = dtransforms.rgb(
    mosa_m, norm="full", coordinates=motors[[CH_PHI, CH_CHI]]
)

fig, ax = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 1]})
ax[0].imshow(rgb_map)
ax[0].set_title("Mosaicity (phi-chi) RGB")
ax[0].axis("off")
ax[1].imshow(
    colorkey, origin="lower", aspect="auto",
    extent=[motors[CH_PHI].min(), motors[CH_PHI].max(),
            motors[CH_CHI].min(), motors[CH_CHI].max()],
)
ax[1].set_xlabel("phi"); ax[1].set_ylabel("chi"); ax[1].set_title("colour key")
plt.tight_layout()
fig.savefig(OUT_DIR / "mosaicity_rgb.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Segment the mosaicity map using `disell`

We use the local `disell` flood-fill cell segmentation
(`disell.flood_fill_dfxm`) on the two-channel mosaicity field. The C++
backend cannot handle NaNs, so invalid pixels are zero-filled in a copy and
the `valid` mask restricts growth. The wrapper lifts 2D `(Y,X,C)` input to
3D internally and squeezes the label image back to `(Y,X)`.

Segmentation thresholds are tunable analysis parameters (Section 1), **not**
final scientific parameters.

In [ ]:
def segment_mosaicity(mosa, valid_mask, *, normalize=SEG_NORMALIZE):
    """Segment a 2-channel mosaicity field into cells with disell.

    Parameters
    ----------
    mosa : ndarray (Y, X, 2)
        phi/chi mosaicity, NaN outside the valid region.
    valid_mask : ndarray (Y, X) bool
    normalize : bool
        If True, per-channel 1-99 pct min-max normalisation (thresholds on
        a ~[0,1] scale). If False, raw angular units are used.

    Returns
    -------
    labels : ndarray (Y, X) int32, 0 = background.
    seg_input : ndarray (Y, X, 2) float32, NaN-free field actually segmented.
    """
    assert mosa.ndim == 3 and mosa.shape[-1] == 2
    assert mosa.shape[:2] == valid_mask.shape

    seg_input = mosa.astype(np.float32).copy()
    if normalize:
        for c in range(seg_input.shape[-1]):
            vals = seg_input[..., c][valid_mask]
            lo, hi = np.nanpercentile(vals, [1, 99])
            if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
                raise ValueError(f"Bad normalisation range for channel {c}: {lo}, {hi}")
            seg_input[..., c] = (seg_input[..., c] - lo) / (hi - lo)
    # C++ backend needs finite values; mask restricts growth regardless.
    seg_input[~np.isfinite(seg_input)] = 0.0

    footprint = np.ones((3, 3), dtype=bool)
    result = disell.flood_fill_dfxm(
        property_map=seg_input,
        footprint=footprint,
        local_threshold=SEG_LOCAL_THRESHOLD,
        global_threshold=SEG_GLOBAL_THRESHOLD,
        footprint_tolerance=SEG_FOOTPRINT_TOLERANCE,
        mask=valid_mask.astype(np.uint8),
        max_iterations=SEG_MAX_ITERATIONS,
        min_grain_size=SEG_MIN_SIZE,
        recycle_small_grains=SEG_RECYCLE_SMALL,
        random_seed=SEG_RANDOM_SEED,
    )
    labels = np.asarray(result["segmentation"], dtype=np.int32)
    return labels, seg_input

In [ ]:
labels, seg_input = segment_mosaicity(mosa_m, valid)
assert labels.shape == valid.shape
n_cells = int(labels.max())
print(f"Number of segmented cells: {n_cells}")
print(f"labelled coverage inside mask: "
      f"{np.count_nonzero((labels>0) & valid)/max(int(valid.sum()),1)*100:.1f} %")

In [ ]:
# Segmentation visual diagnostics.
rng = np.random.default_rng(0)
v = np.nanpercentile(np.abs(strain_m), 98)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0, 0].imshow(np.where(labels > 0, labels, np.nan), cmap="nipy_spectral",
                  interpolation="nearest")
axes[0, 0].set_title(f"Segmentation labels (n={n_cells})")
axes[0, 0].axis("off")

axes[0, 1].imshow(rgb_map)
axes[0, 1].imshow(np.where(labels > 0, labels, np.nan), cmap="nipy_spectral",
                  interpolation="nearest", alpha=0.35)
axes[0, 1].set_title("labels over mosaicity RGB")
axes[0, 1].axis("off")

# Boundaries computed in the next section; precompute a quick version here.
_b = np.zeros_like(labels, dtype=bool)
_b[:-1, :] |= labels[:-1, :] != labels[1:, :]
_b[:, :-1] |= labels[:, :-1] != labels[:, 1:]
_b &= valid & (labels > 0)

axes[1, 0].imshow(rgb_map)
axes[1, 0].imshow(np.where(_b, 1.0, np.nan), cmap="gray", interpolation="nearest")
axes[1, 0].set_title("boundaries over mosaicity RGB")
axes[1, 0].axis("off")

axes[1, 1].imshow(strain_m, cmap="RdBu_r", vmin=-v, vmax=v)
axes[1, 1].imshow(np.where(_b, 1.0, np.nan), cmap="gray", interpolation="nearest")
axes[1, 1].set_title("boundaries over corrected strain")
axes[1, 1].axis("off")

plt.tight_layout()
fig.savefig(OUT_DIR / "segmentation_labels.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Boundary extraction

Extract the cell-boundary mask and the complementary interior mask, both
restricted to valid, labelled pixels.

In [ ]:
if find_boundaries is not None:
    boundary_mask = find_boundaries(labels, mode="outer")
else:
    boundary_mask = np.zeros_like(labels, dtype=bool)
    boundary_mask[:-1, :] |= labels[:-1, :] != labels[1:, :]
    boundary_mask[:, :-1] |= labels[:, :-1] != labels[:, 1:]

boundary_mask &= valid
boundary_mask &= labels > 0

interior_mask = valid & (labels > 0) & ~boundary_mask

print('boundary pixels :', int(boundary_mask.sum()))
print('interior pixels :', int(interior_mask.sum()))

np.save(OUT_DIR / "labels.npy", labels)
np.save(OUT_DIR / "boundary_mask.npy", boundary_mask)

## 8. Compare segmented boundaries with strain

Compare signed strain and absolute/deviation strain at cell boundaries vs
cell interiors, plus a distance-to-boundary trend. Deviation strain is
`|strain - median(strain)|` over the valid region.

In [ ]:
def summarise_values(name, values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return {"name": name, "n": 0, "mean": np.nan, "median": np.nan,
                "std": np.nan, "mad": np.nan, "p05": np.nan, "p95": np.nan}
    return {
        "name": name,
        "n": values.size,
        "mean": np.mean(values),
        "median": np.median(values),
        "std": np.std(values),
        "mad": np.median(np.abs(values - np.median(values))),
        "p05": np.percentile(values, 5),
        "p95": np.percentile(values, 95),
    }


def finite_pair(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    return a[m], b[m]


def cohens_d(a, b):
    a = np.asarray(a); a = a[np.isfinite(a)]
    b = np.asarray(b); b = b[np.isfinite(b)]
    if a.size < 2 or b.size < 2:
        return np.nan
    na, nb = a.size, b.size
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    if sp == 0:
        return np.nan
    return (a.mean() - b.mean()) / sp


def robust_median_diff(a, b):
    """MAD-normalised median difference (median(a)-median(b)) / pooled MAD."""
    a = np.asarray(a); a = a[np.isfinite(a)]
    b = np.asarray(b); b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0:
        return np.nan
    mad_a = np.median(np.abs(a - np.median(a)))
    mad_b = np.median(np.abs(b - np.median(b)))
    pooled = 0.5 * (mad_a + mad_b)
    if pooled == 0:
        return np.nan
    return (np.median(a) - np.median(b)) / pooled

In [ ]:
strain_ref = np.nanmedian(strain_m[valid])
abs_strain_m = np.abs(strain_m - strain_ref)

strain_boundary = strain_m[boundary_mask]
strain_interior = strain_m[interior_mask]
abs_strain_boundary = abs_strain_m[boundary_mask]
abs_strain_interior = abs_strain_m[interior_mask]

summary_rows = [
    summarise_values("signed strain @ boundary", strain_boundary),
    summarise_values("signed strain @ interior", strain_interior),
    summarise_values("abs/dev strain @ boundary", abs_strain_boundary),
    summarise_values("abs/dev strain @ interior", abs_strain_interior),
]
boundary_strain_summary = pd.DataFrame(summary_rows)
boundary_strain_summary.to_csv(OUT_DIR / "boundary_strain_summary.csv", index=False)

diff_mean_signed = np.nanmean(strain_boundary) - np.nanmean(strain_interior)
diff_median_signed = np.nanmedian(strain_boundary) - np.nanmedian(strain_interior)
diff_mean_abs = np.nanmean(abs_strain_boundary) - np.nanmean(abs_strain_interior)
diff_median_abs = np.nanmedian(abs_strain_boundary) - np.nanmedian(abs_strain_interior)
d_signed = cohens_d(strain_boundary, strain_interior)
d_abs = cohens_d(abs_strain_boundary, abs_strain_interior)
rmd_abs = robust_median_diff(abs_strain_boundary, abs_strain_interior)

print(boundary_strain_summary.to_string(index=False))
print()
print(f"signed strain: boundary-interior mean   = {diff_mean_signed:.3e}")
print(f"signed strain: boundary-interior median = {diff_median_signed:.3e}")
print(f"abs/dev strain: boundary-interior mean   = {diff_mean_abs:.3e}")
print(f"abs/dev strain: boundary-interior median = {diff_median_abs:.3e}")
print(f"Cohen's d (signed, boundary vs interior)   = {d_signed:.3f}")
print(f"Cohen's d (abs/dev, boundary vs interior)  = {d_abs:.3f}")
print(f"MAD-normalised median diff (abs/dev)       = {rmd_abs:.3f}")

In [ ]:
# Histograms: boundary vs interior.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

b1 = strain_boundary[np.isfinite(strain_boundary)]
i1 = strain_interior[np.isfinite(strain_interior)]
lo, hi = np.nanpercentile(np.concatenate([b1, i1]), [1, 99])
bins = np.linspace(lo, hi, 60)
axes[0].hist(i1, bins=bins, density=True, alpha=0.5, label="interior")
axes[0].hist(b1, bins=bins, density=True, alpha=0.5, label="boundary")
axes[0].set_title("signed strain: boundary vs interior")
axes[0].set_xlabel("strain"); axes[0].set_ylabel("density"); axes[0].legend()

b2 = abs_strain_boundary[np.isfinite(abs_strain_boundary)]
i2 = abs_strain_interior[np.isfinite(abs_strain_interior)]
hi2 = np.nanpercentile(np.concatenate([b2, i2]), 99)
bins2 = np.linspace(0, hi2, 60)
axes[1].hist(i2, bins=bins2, density=True, alpha=0.5, label="interior")
axes[1].hist(b2, bins=bins2, density=True, alpha=0.5, label="boundary")
axes[1].set_title("abs/dev strain: boundary vs interior")
axes[1].set_xlabel("|strain - median|"); axes[1].set_ylabel("density"); axes[1].legend()

plt.tight_layout()
fig.savefig(OUT_DIR / "boundaries_on_strain.png", dpi=200, bbox_inches="tight")
plt.show()

### Distance-to-boundary vs strain

Distance (in pixels) from each valid labelled pixel to the nearest cell
boundary, correlated with signed and absolute/deviation strain.

In [ ]:
dist_to_boundary = distance_transform_edt(~boundary_mask)
dist_to_boundary = np.where(valid & (labels > 0), dist_to_boundary, np.nan)

x, y = finite_pair(dist_to_boundary, strain_m)
pear_d_signed = stats.pearsonr(x, y)
spear_d_signed = stats.spearmanr(x, y)
xa, ya = finite_pair(dist_to_boundary, abs_strain_m)
pear_d_abs = stats.pearsonr(xa, ya)
spear_d_abs = stats.spearmanr(xa, ya)

print(f"dist-to-boundary vs signed strain : Pearson r={pear_d_signed[0]:.3f}, "
      f"Spearman rho={spear_d_signed[0]:.3f} (n={x.size})")
print(f"dist-to-boundary vs abs/dev strain: Pearson r={pear_d_abs[0]:.3f}, "
      f"Spearman rho={spear_d_abs[0]:.3f} (n={xa.size})")
print('NOTE: pixel-wise p-values are inflated by spatial autocorrelation; treat as descriptive.')

In [ ]:
def binned_statistic_xy(x, y, bins=30):
    x = np.asarray(x); y = np.asarray(y)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]; y = y[m]
    edges = np.linspace(np.nanmin(x), np.nanmax(x), bins + 1)
    centres = 0.5 * (edges[:-1] + edges[1:])
    mean = np.full(bins, np.nan)
    median = np.full(bins, np.nan)
    count = np.zeros(bins, dtype=int)
    for i in range(bins):
        mm = (x >= edges[i]) & (x < edges[i + 1])
        if np.any(mm):
            mean[i] = np.nanmean(y[mm])
            median[i] = np.nanmedian(y[mm])
            count[i] = int(np.sum(mm))
    return centres, mean, median, count

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hexbin(x, y, gridsize=50, cmap="viridis", mincnt=1)
axes[0].set_xlabel("distance to boundary (px)"); axes[0].set_ylabel("signed strain")
axes[0].set_title(f"strain vs dist (rho={spear_d_signed[0]:.3f})")

axes[1].hexbin(xa, ya, gridsize=50, cmap="viridis", mincnt=1)
axes[1].set_xlabel("distance to boundary (px)"); axes[1].set_ylabel("abs/dev strain")
axes[1].set_title(f"abs/dev strain vs dist (rho={spear_d_abs[0]:.3f})")

cen, mn, md_, ct = binned_statistic_xy(xa, ya, bins=30)
axes[2].plot(cen, md_, "-o", ms=3)
axes[2].set_xlabel("distance to boundary (px)"); axes[2].set_ylabel("median abs/dev strain")
axes[2].set_title("binned median abs/dev strain vs dist")

plt.tight_layout()
fig.savefig(OUT_DIR / "distance_to_boundary_vs_strain.png", dpi=200, bbox_inches="tight")
# second dedicated abs figure (per spec filename)
fig2, ax2 = plt.subplots(figsize=(7, 5))
ax2.hexbin(xa, ya, gridsize=50, cmap="viridis", mincnt=1)
ax2.set_xlabel("distance to boundary (px)"); ax2.set_ylabel("abs/dev strain")
ax2.set_title(f"abs/dev strain vs dist (rho={spear_d_abs[0]:.3f})")
fig2.savefig(OUT_DIR / "distance_to_boundary_vs_abs_strain.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Compute KAM using `darling.transforms.kam`

KAM = kernel average misorientation of the 2-channel (phi, chi) mosaicity
field. Inspect the signature, then call it on `mosa_m` with the same valid
mask used everywhere else.

**Interpretation:** here KAM is a *local angular / motor-space
mosaicity-gradient* measure derived from the DFXM mosaicity map. It is a
projected quantity (the full rotation matrix is unknown) and should **not**
be read as a full crystallographic misorientation unless a proper geometry
conversion is applied.

In [ ]:
print(dtransforms.kam)
print('signature:', inspect.signature(dtransforms.kam))

kam_m = dtransforms.kam(mosa_m, size=KAM_SIZE)
kam_m = np.asarray(kam_m, dtype=float)
assert kam_m.shape == valid.shape, (kam_m.shape, valid.shape)
kam_m = np.where(valid, kam_m, np.nan)

np.save(OUT_DIR / "kam.npy", kam_m)
print('kam finite pixels:', int(np.isfinite(kam_m).sum()))
print('kam 2-98 pct:', np.nanpercentile(kam_m, [2, 98]))

In [ ]:
kv = np.nanpercentile(kam_m, 98)
sv = np.nanpercentile(np.abs(strain_m), 98)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
im0 = axes[0].imshow(kam_m, cmap="plasma", vmax=kv)
axes[0].set_title("KAM (motor-space mosaicity gradient)"); axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].imshow(kam_m, cmap="plasma", vmax=kv)
axes[1].imshow(np.where(boundary_mask, 1.0, np.nan), cmap="gray", interpolation="nearest")
axes[1].set_title("KAM + segmentation boundaries"); axes[1].axis("off")

im2 = axes[2].imshow(strain_m, cmap="RdBu_r", vmin=-sv, vmax=sv)
axes[2].set_title("corrected strain"); axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
fig.savefig(OUT_DIR / "kam_map.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Compare KAM with strain

Pixel-wise correlations and binned trends of KAM against signed and
absolute/deviation strain.

In [ ]:
kam_vals, strain_vals = finite_pair(kam_m, strain_m)
kam_vals_abs, abs_strain_vals = finite_pair(kam_m, abs_strain_m)

pear_kam_signed = stats.pearsonr(kam_vals, strain_vals)
spear_kam_signed = stats.spearmanr(kam_vals, strain_vals)
pear_kam_abs = stats.pearsonr(kam_vals_abs, abs_strain_vals)
spear_kam_abs = stats.spearmanr(kam_vals_abs, abs_strain_vals)

print(f"KAM vs signed strain : Pearson r={pear_kam_signed[0]:.3f}, "
      f"Spearman rho={spear_kam_signed[0]:.3f} (n={kam_vals.size})")
print(f"KAM vs abs/dev strain: Pearson r={pear_kam_abs[0]:.3f}, "
      f"Spearman rho={spear_kam_abs[0]:.3f} (n={kam_vals_abs.size})")
print('NOTE: pixel-wise p-values inflated by spatial autocorrelation; descriptive only.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].hexbin(strain_vals, kam_vals, gridsize=60, cmap="viridis", mincnt=1)
cen, mn, md_, ct = binned_statistic_xy(strain_vals, kam_vals, bins=30)
axes[0].plot(cen, md_, "r-o", ms=3, label="binned median")
axes[0].set_xlabel("signed strain"); axes[0].set_ylabel("KAM")
axes[0].set_title(f"KAM vs signed strain (rho={spear_kam_signed[0]:.3f})"); axes[0].legend()

axes[1].hexbin(abs_strain_vals, kam_vals_abs, gridsize=60, cmap="viridis", mincnt=1)
cen2, mn2, md2, ct2 = binned_statistic_xy(abs_strain_vals, kam_vals_abs, bins=30)
axes[1].plot(cen2, md2, "r-o", ms=3, label="binned median")
axes[1].set_xlabel("abs/dev strain"); axes[1].set_ylabel("KAM")
axes[1].set_title(f"KAM vs abs/dev strain (rho={spear_kam_abs[0]:.3f})"); axes[1].legend()

plt.tight_layout()
axes[0].figure.savefig(OUT_DIR / "kam_vs_strain.png", dpi=200, bbox_inches="tight")
fig_abs, ax_abs = plt.subplots(figsize=(7, 6))
ax_abs.hexbin(abs_strain_vals, kam_vals_abs, gridsize=60, cmap="viridis", mincnt=1)
ax_abs.plot(cen2, md2, "r-o", ms=3, label="binned median")
ax_abs.set_xlabel("abs/dev strain"); ax_abs.set_ylabel("KAM")
ax_abs.set_title(f"KAM vs abs/dev strain (rho={spear_kam_abs[0]:.3f})"); ax_abs.legend()
fig_abs.savefig(OUT_DIR / "kam_vs_abs_strain.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Per-cell statistics

Aggregate mosaicity, strain and KAM statistics per segmented cell and save
the table to CSV.

In [ ]:
rows = []
for lab in np.unique(labels):
    if lab == 0:
        continue
    region = valid & (labels == lab)
    if not np.any(region):
        continue
    region_boundary = region & boundary_mask
    rows.append({
        "label": int(lab),
        "area_px": int(np.sum(region)),
        "mean_phi": np.nanmean(phi_m[region]),
        "mean_chi": np.nanmean(chi_m[region]),
        "std_phi": np.nanstd(phi_m[region]),
        "std_chi": np.nanstd(chi_m[region]),
        "mean_strain": np.nanmean(strain_m[region]),
        "median_strain": np.nanmedian(strain_m[region]),
        "std_strain": np.nanstd(strain_m[region]),
        "mean_abs_strain": np.nanmean(abs_strain_m[region]),
        "median_abs_strain": np.nanmedian(abs_strain_m[region]),
        "mean_kam": np.nanmean(kam_m[region]),
        "median_kam": np.nanmedian(kam_m[region]),
        "boundary_px": int(np.sum(region_boundary)),
        "boundary_fraction": float(np.sum(region_boundary) / np.sum(region)),
    })

cell_stats = pd.DataFrame(rows)
cell_stats.to_csv(OUT_DIR / "cell_stats.csv", index=False)
print(f"per-cell table: {len(cell_stats)} cells")
cell_stats.head()

In [ ]:
if len(cell_stats):
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes[0, 0].scatter(cell_stats["area_px"], cell_stats["mean_abs_strain"], s=8, alpha=0.5)
    axes[0, 0].set_xscale("log")
    axes[0, 0].set_xlabel("cell area (px)"); axes[0, 0].set_ylabel("mean abs/dev strain")
    axes[0, 0].set_title("cell area vs mean abs/dev strain")

    axes[0, 1].scatter(cell_stats["area_px"], cell_stats["std_strain"], s=8, alpha=0.5)
    axes[0, 1].set_xscale("log")
    axes[0, 1].set_xlabel("cell area (px)"); axes[0, 1].set_ylabel("strain std")
    axes[0, 1].set_title("cell area vs strain std")

    axes[0, 2].scatter(cell_stats["mean_kam"], cell_stats["mean_abs_strain"], s=8, alpha=0.5)
    axes[0, 2].set_xlabel("mean KAM"); axes[0, 2].set_ylabel("mean abs/dev strain")
    axes[0, 2].set_title("mean KAM vs mean abs/dev strain")

    axes[1, 0].scatter(cell_stats["mean_kam"], cell_stats["std_strain"], s=8, alpha=0.5)
    axes[1, 0].set_xlabel("mean KAM"); axes[1, 0].set_ylabel("strain std")
    axes[1, 0].set_title("mean KAM vs strain std")

    axes[1, 1].scatter(cell_stats["boundary_fraction"], cell_stats["mean_abs_strain"], s=8, alpha=0.5)
    axes[1, 1].set_xlabel("boundary fraction"); axes[1, 1].set_ylabel("mean abs/dev strain")
    axes[1, 1].set_title("boundary fraction vs mean abs/dev strain")

    axes[1, 2].axis("off")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "cell_area_vs_abs_strain.png", dpi=200, bbox_inches="tight")
    # dedicated KAM-vs-abs-strain per-cell figure (per spec filename)
    figk, axk = plt.subplots(figsize=(7, 6))
    axk.scatter(cell_stats["mean_kam"], cell_stats["mean_abs_strain"], s=8, alpha=0.5)
    axk.set_xlabel("mean KAM"); axk.set_ylabel("mean abs/dev strain")
    axk.set_title("per-cell mean KAM vs mean abs/dev strain")
    figk.savefig(OUT_DIR / "cell_kam_vs_abs_strain.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No cells in table; skipping per-cell plots.")

## 12. Optional second 2D mosaicity dataset

A second 2D mosaicity dataset from the **same ROI** at a slightly different
strain/load step. It may contain mosaicity only (no strain). If
`DATA_DIR_2` does not exist, every cell in Sections 12–14 is a clean no-op.

In [ ]:
def load_2d_mosaicity_dataset(path, ch_phi=0, ch_chi=1):
    """Load a 2D mosaicity dataset.

    Expected possibilities:
    - mean.npy with shape (Y, X, >=2)
    - mean.npy with shape (Y, X, 2)
    - optional motors.npy
    - optional metadata JSON

    Returns
    -------
    mosa : ndarray (Y, X, 2), phi/chi.
    valid : ndarray (Y, X) bool.
    metadata : dict
    """
    path = Path(path)
    mean_path = path / "mean.npy"
    if not mean_path.exists():
        raise FileNotFoundError(f"No mean.npy in {path}")
    arr = np.load(mean_path).astype(float)
    if arr.ndim != 3 or arr.shape[-1] < 2:
        raise ValueError(f"Expected mean.npy with shape (Y,X,>=2), got {arr.shape}")

    phi = arr[..., ch_phi]
    chi = arr[..., ch_chi]
    mosa = np.stack([phi, chi], axis=-1)

    valid = np.isfinite(phi) & np.isfinite(chi)
    valid &= ~((phi == 0) & (chi == 0))  # peaks leaves unlabelled pixels at 0
    mosa[~valid] = np.nan

    metadata = {}
    motors_path = path / "motors.npy"
    if motors_path.exists():
        metadata["motors"] = np.load(motors_path)
    info_path = path / "processing_info.json"
    if info_path.exists():
        try:
            metadata["processing_info"] = json.loads(info_path.read_text())
        except json.JSONDecodeError:
            metadata["processing_info"] = None
    return mosa, valid, metadata

In [ ]:
mosa_2 = valid_2 = phi_2 = chi_2 = rgb_2 = kam_2 = labels_2 = boundary_2 = None
meta_2 = {}

if HAS_SECOND_DATASET:
    mosa_2, valid_2, meta_2 = load_2d_mosaicity_dataset(DATA_DIR_2, ch_phi=CH_PHI, ch_chi=CH_CHI)
    phi_2 = mosa_2[..., 0]
    chi_2 = mosa_2[..., 1]
    print("dataset 2 mosa:", mosa_2.shape, "valid frac:", round(float(valid_2.mean()), 3))

    # RGB (use motor coordinates if available, else dynamic norm).
    motors_2 = meta_2.get("motors")
    if motors_2 is not None and motors_2.shape[0] >= 2:
        rgb_2, _, _ = dtransforms.rgb(mosa_2, norm="full",
                                      coordinates=motors_2[[CH_PHI, CH_CHI]])
    else:
        rgb_2, _, _ = dtransforms.rgb(mosa_2, norm="dynamic")

    kam_2 = np.asarray(dtransforms.kam(mosa_2, size=KAM_SIZE), dtype=float)
    kam_2 = np.where(valid_2, kam_2, np.nan)

    labels_2, _ = segment_mosaicity(mosa_2, valid_2)
    if find_boundaries is not None:
        boundary_2 = find_boundaries(labels_2, mode="outer")
    else:
        boundary_2 = np.zeros_like(labels_2, dtype=bool)
        boundary_2[:-1, :] |= labels_2[:-1, :] != labels_2[1:, :]
        boundary_2[:, :-1] |= labels_2[:, :-1] != labels_2[:, 1:]
    boundary_2 &= valid_2 & (labels_2 > 0)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb_2)
    ax.set_title("Mosaicity RGB - dataset 2"); ax.axis("off")
    fig.savefig(OUT_DIR / "mosaicity_rgb_dataset_2.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("HAS_SECOND_DATASET is False -> skipping dataset-2 load.")

## 13. Registration / alignment of the two maps

Initially assume the two datasets share the same ROI and pixel grid. If the
shapes match we proceed directly; otherwise we estimate a rigid translation
with `phase_cross_correlation` (only translation — no non-rigid warping).

> **Registration must be verified before interpreting microstructure-change
> correlations.** If registration is uncertain, the Section-14 results are
> preliminary.

In [ ]:
registration_shift = (0.0, 0.0)
registration_ok = True
registration_note = ""

if HAS_SECOND_DATASET:
    same_shape = mosa_m.shape == mosa_2.shape
    print("same_shape:", same_shape)
    if not same_shape:
        registration_ok = False
        registration_note = (
            f"Shape mismatch {mosa_m.shape} vs {mosa_2.shape}: cropping/registration "
            "required. Section 14 results are PRELIMINARY.")
        print(registration_note)
    elif phase_cross_correlation is not None:
        # Register on the KAM signal (robust to colour scaling).
        ref = np.nan_to_num(kam_m)
        mov = np.nan_to_num(kam_2)
        shift_est, error, _ = phase_cross_correlation(ref, mov, upsample_factor=10)
        registration_shift = tuple(float(s) for s in shift_est)
        print("estimated (row, col) shift dataset2->dataset1:", registration_shift)
        print("phase-correlation error:", float(error))
        if np.hypot(*registration_shift) > 5:
            registration_note = (
                "Estimated shift > 5 px: verify registration before trusting "
                "microstructure-change correlations.")
            print(registration_note)
    else:
        registration_note = "phase_cross_correlation unavailable; assuming aligned grids."
        print(registration_note)
else:
    print("No second dataset -> registration skipped.")

In [ ]:
# Apply the estimated translation to dataset-2 fields if it is non-trivial
# and the grids match. We shift mosaicity channels and KAM, then rebuild the
# valid mask and re-segment so labels_2 / boundary_2 stay consistent.
if HAS_SECOND_DATASET and registration_ok and np.hypot(*registration_shift) > 0.5:
    from scipy.ndimage import shift as ndi_shift
    sr, sc = registration_shift
    phi_2 = ndi_shift(np.nan_to_num(phi_2), (sr, sc), order=1, cval=np.nan)
    chi_2 = ndi_shift(np.nan_to_num(chi_2), (sr, sc), order=1, cval=np.nan)
    mosa_2 = np.stack([phi_2, chi_2], axis=-1)
    valid_2 = np.isfinite(phi_2) & np.isfinite(chi_2)
    mosa_2[~valid_2] = np.nan
    kam_2 = np.where(valid_2, np.asarray(dtransforms.kam(mosa_2, size=KAM_SIZE), dtype=float), np.nan)
    labels_2, _ = segment_mosaicity(mosa_2, valid_2)
    if find_boundaries is not None:
        boundary_2 = find_boundaries(labels_2, mode="outer") & valid_2 & (labels_2 > 0)
    print("applied registration shift and re-segmented dataset 2.")
elif HAS_SECOND_DATASET:
    print("No registration shift applied (sub-pixel or grids differ / unavailable).")
else:
    print("No second dataset -> nothing to align.")

## 14. Microstructure change vs initial strain

Where both datasets are valid, compute mosaicity change `delta_mosa`,
KAM change `abs_delta_kam`, and boundary changes, then correlate them with
the **initial** corrected strain field. Tests whether high/low strain
regions are the regions that change most between load steps.

In [ ]:
delta_mosa = delta_kam = abs_delta_kam = boundary_change = None
delta_dist_boundary = valid_12 = None
second_corr = {}

if HAS_SECOND_DATASET and mosa_m.shape == mosa_2.shape:
    valid_12 = valid & valid_2

    delta_phi = phi_2 - phi_m
    delta_chi = chi_2 - chi_m
    delta_mosa = np.sqrt(delta_phi**2 + delta_chi**2)
    delta_mosa = np.where(valid_12, delta_mosa, np.nan)

    delta_kam = kam_2 - kam_m
    abs_delta_kam = np.abs(delta_kam)
    abs_delta_kam = np.where(valid_12, abs_delta_kam, np.nan)

    boundary_change = boundary_mask ^ boundary_2
    boundary_change &= valid_12

    dist_boundary_1 = distance_transform_edt(~boundary_mask)
    dist_boundary_2 = distance_transform_edt(~boundary_2)
    delta_dist_boundary = dist_boundary_2 - dist_boundary_1
    delta_dist_boundary = np.where(valid_12, delta_dist_boundary, np.nan)

    # Correlations of initial strain with later change.
    xa, ya = finite_pair(strain_m, delta_mosa)
    second_corr["strain_vs_delta_mosa"] = (stats.pearsonr(xa, ya)[0], stats.spearmanr(xa, ya)[0])
    xb, yb = finite_pair(abs_strain_m, delta_mosa)
    second_corr["abs_strain_vs_delta_mosa"] = (stats.pearsonr(xb, yb)[0], stats.spearmanr(xb, yb)[0])
    xc, yc = finite_pair(strain_m, abs_delta_kam)
    second_corr["strain_vs_abs_delta_kam"] = (stats.pearsonr(xc, yc)[0], stats.spearmanr(xc, yc)[0])
    xd, yd = finite_pair(abs_strain_m, abs_delta_kam)
    second_corr["abs_strain_vs_abs_delta_kam"] = (stats.pearsonr(xd, yd)[0], stats.spearmanr(xd, yd)[0])

    abs_strain_changed = abs_strain_m[boundary_change & valid_12]
    abs_strain_unchanged = abs_strain_m[(~boundary_change) & valid_12]
    med_changed = np.nanmedian(abs_strain_changed)
    med_unchanged = np.nanmedian(abs_strain_unchanged)

    for k, (pr, sp) in second_corr.items():
        print(f"{k:32s}: Pearson r={pr:.3f}, Spearman rho={sp:.3f}")
    print(f"median abs/dev strain @ boundary-change   : {med_changed:.3e}")
    print(f"median abs/dev strain @ unchanged         : {med_unchanged:.3e}")
    print('NOTE: depends on reliable ROI registration; treat as preliminary if uncertain.')
elif HAS_SECOND_DATASET:
    print("Dataset shapes differ; skipping change correlations (registration needed).")
else:
    print("No second dataset -> Section 14 skipped.")

In [ ]:
if HAS_SECOND_DATASET and delta_mosa is not None:
    sv = np.nanpercentile(np.abs(strain_m), 98)
    dv = np.nanpercentile(delta_mosa, 98)
    fig, axes = plt.subplots(2, 4, figsize=(22, 11))
    axes[0, 0].imshow(rgb_map); axes[0, 0].set_title("dataset 1 RGB"); axes[0, 0].axis("off")
    axes[0, 1].imshow(rgb_2); axes[0, 1].set_title("dataset 2 RGB"); axes[0, 1].axis("off")
    im = axes[0, 2].imshow(delta_mosa, cmap="magma", vmax=dv)
    axes[0, 2].set_title("delta_mosa"); axes[0, 2].axis("off")
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)
    im = axes[0, 3].imshow(strain_m, cmap="RdBu_r", vmin=-sv, vmax=sv)
    axes[0, 3].set_title("initial corrected strain"); axes[0, 3].axis("off")
    plt.colorbar(im, ax=axes[0, 3], fraction=0.046, pad=0.04)

    xa, ya = finite_pair(strain_m, delta_mosa)
    axes[1, 0].hexbin(xa, ya, gridsize=50, cmap="viridis", mincnt=1)
    axes[1, 0].set_xlabel("initial signed strain"); axes[1, 0].set_ylabel("delta_mosa")
    axes[1, 0].set_title("delta_mosa vs initial signed strain")

    xb, yb = finite_pair(abs_strain_m, delta_mosa)
    axes[1, 1].hexbin(xb, yb, gridsize=50, cmap="viridis", mincnt=1)
    cen, mn, md_, ct = binned_statistic_xy(xb, yb, bins=30)
    axes[1, 1].plot(cen, md_, "r-o", ms=3)
    axes[1, 1].set_xlabel("initial abs/dev strain"); axes[1, 1].set_ylabel("delta_mosa")
    axes[1, 1].set_title("delta_mosa vs initial abs/dev strain")

    axes[1, 2].imshow(strain_m, cmap="RdBu_r", vmin=-sv, vmax=sv)
    axes[1, 2].imshow(np.where(boundary_change, 1.0, np.nan), cmap="gray", interpolation="nearest")
    axes[1, 2].set_title("boundary change on initial strain"); axes[1, 2].axis("off")

    axes[1, 3].plot(cen, md_, "b-o", ms=3)
    axes[1, 3].set_xlabel("initial abs/dev strain"); axes[1, 3].set_ylabel("binned median delta_mosa")
    axes[1, 3].set_title("binned delta_mosa vs initial abs/dev strain")

    plt.tight_layout()
    fig.savefig(OUT_DIR / "delta_mosa.png", dpi=200, bbox_inches="tight")
    for nm, (xx, yy) in {
        "delta_mosa_vs_initial_strain.png": (xa, ya),
        "delta_mosa_vs_initial_abs_strain.png": (xb, yb),
    }.items():
        f2, a2 = plt.subplots(figsize=(7, 5))
        a2.hexbin(xx, yy, gridsize=50, cmap="viridis", mincnt=1)
        a2.set_xlabel("initial strain"); a2.set_ylabel("delta_mosa")
        f2.savefig(OUT_DIR / nm, dpi=200, bbox_inches="tight")
        plt.close(f2)
    fbc, abc = plt.subplots(figsize=(7, 6))
    abc.imshow(strain_m, cmap="RdBu_r", vmin=-sv, vmax=sv)
    abc.imshow(np.where(boundary_change, 1.0, np.nan), cmap="gray", interpolation="nearest")
    abc.set_title("boundary change on initial strain"); abc.axis("off")
    fbc.savefig(OUT_DIR / "boundary_change_on_initial_strain.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("No second-dataset change maps to plot.")

## 15. Save outputs

In [ ]:
# Core arrays already saved alongside their sections (labels, boundary_mask,
# kam). Save the remaining first-dataset arrays and (if present) the
# second-dataset change arrays.
np.save(OUT_DIR / "strain_m.npy", strain_m)
np.save(OUT_DIR / "abs_strain_m.npy", abs_strain_m)
np.save(OUT_DIR / "valid.npy", valid)
np.save(OUT_DIR / "dist_to_boundary.npy", dist_to_boundary)

if HAS_SECOND_DATASET and delta_mosa is not None:
    np.save(OUT_DIR / "delta_mosa.npy", delta_mosa)
    np.save(OUT_DIR / "delta_kam.npy", delta_kam)
    np.save(OUT_DIR / "boundary_change.npy", boundary_change)

print("Saved arrays to:", OUT_DIR)
print(sorted(p.name for p in OUT_DIR.glob('*.npy')))
print(sorted(p.name for p in OUT_DIR.glob('*.png')))
print(sorted(p.name for p in OUT_DIR.glob('*.csv')))

## 16. Final numerical summary and caveats

In [ ]:
n_valid = int(valid.sum())
n_boundary = int(boundary_mask.sum())
n_interior = int(interior_mask.sum())

lines = []
lines.append(f"Number of valid pixels:    {n_valid}")
lines.append(f"Number of segmented cells: {n_cells}")
lines.append(f"Boundary pixels:           {n_boundary}")
lines.append(f"Interior pixels:           {n_interior}")
lines.append("")
lines.append(f"Boundary mean strain:              {np.nanmean(strain_boundary):.3e}")
lines.append(f"Interior mean strain:              {np.nanmean(strain_interior):.3e}")
lines.append(f"Boundary - interior mean strain:   {diff_mean_signed:.3e}")
lines.append("")
lines.append(f"Boundary median abs/dev strain:            {np.nanmedian(abs_strain_boundary):.3e}")
lines.append(f"Interior median abs/dev strain:            {np.nanmedian(abs_strain_interior):.3e}")
lines.append(f"Boundary - interior median abs/dev strain: {diff_median_abs:.3e}")
lines.append("")
lines.append(f"Spearman KAM vs strain:           {spear_kam_signed[0]:.3f}")
lines.append(f"Spearman KAM vs abs/dev strain:   {spear_kam_abs[0]:.3f}")
lines.append("")
lines.append(f"Spearman dist-to-boundary vs strain:         {spear_d_signed[0]:.3f}")
lines.append(f"Spearman dist-to-boundary vs abs/dev strain: {spear_d_abs[0]:.3f}")

if HAS_SECOND_DATASET and delta_mosa is not None:
    n_bc = int((boundary_change & valid_12).sum())
    n_nbc = int(((~boundary_change) & valid_12).sum())
    lines.append("")
    lines.append("Second dataset loaded: yes")
    lines.append(f"Spearman initial strain vs delta_mosa:           {second_corr['strain_vs_delta_mosa'][1]:.3f}")
    lines.append(f"Spearman initial abs/dev strain vs delta_mosa:   {second_corr['abs_strain_vs_delta_mosa'][1]:.3f}")
    lines.append(f"Spearman initial abs/dev strain vs abs_delta_kam:{second_corr['abs_strain_vs_abs_delta_kam'][1]:.3f}")
    lines.append(f"Boundary-change pixels:     {n_bc}")
    lines.append(f"Non-boundary-change pixels: {n_nbc}")
    lines.append(f"Median initial abs/dev strain @ boundary-change: {np.nanmedian(abs_strain_m[boundary_change & valid_12]):.3e}")
    lines.append(f"Median initial abs/dev strain @ unchanged:       {np.nanmedian(abs_strain_m[(~boundary_change) & valid_12]):.3e}")
    if registration_note:
        lines.append(f"Registration note: {registration_note}")
else:
    lines.append("")
    lines.append("Second dataset loaded: no")

summary_text = "\n".join(lines)
print(summary_text)
(OUT_DIR / "summary.txt").write_text(summary_text)

## Caveats

- The corrected strain field is inherited from the existing 2θ / obpitch
  correction workflow. This notebook does not rederive that correction.
- The strain field is relative unless an absolute reference angle is supplied.
- The empirical ramp/geometric correction may remove a real long-wavelength
  strain gradient. Interpret spatial trends accordingly.
- KAM computed from `darling.transforms.kam` on the DFXM mosaicity map is a
  local angular/motor-space mosaicity-gradient measure. It should not be
  overinterpreted as full crystallographic misorientation unless the proper
  geometry conversion has been applied.
- Segmentation boundaries depend on the chosen segmentation thresholds
  (`SEG_*` parameters). Re-run with different thresholds to test stability.
- Pixel-wise correlations have inflated sample sizes because neighbouring
  pixels are spatially autocorrelated. Treat p-values as descriptive only.
- Comparison between two load steps requires reliable registration of the
  same ROI. If registration is uncertain, microstructure-change correlations
  are preliminary.